In [ ]:
import sys
import pathlib
from collections import Counter
from tqdm.notebook import tqdm
import subprocess

import numpy as np
from scipy.sparse import csr_matrix
import pandas as pd
import anndata as ad

In [ ]:
import yaml

base_path = pathlib.Path('../..').resolve()
sys.path.append(str(base_path))
from helpers import singlecell_utils

with open(base_path / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

In [ ]:
GENE_H5AD = base_path / cfg['gene_h5ad']
PAS_H5AD = pathlib.Path('../singlecell_counts/') / GENE_H5AD.name.replace('.h5ad', '_PAS.h5ad')
COUNTS_DIR = '../counts/'

CURRENT_LEVEL = 'Channel'
MERGED_LEVEL = 'individualID'

# Read the entire barcode table to get the list of files and cells

In [ ]:
ad_pas = singlecell_utils.read_everything_but_X(PAS_H5AD)

In [ ]:
df_obs = ad_pas.obs
df_obs.head()

# Donor - Channel

In [ ]:
df_sc = df_obs[[MERGED_LEVEL, CURRENT_LEVEL]].drop_duplicates(subset=[MERGED_LEVEL, CURRENT_LEVEL])
cur_2_merged = dict(zip(df_sc[CURRENT_LEVEL], df_sc[MERGED_LEVEL]))
len(cur_2_merged)

In [ ]:
df_sc.groupby(MERGED_LEVEL, observed=True).size().value_counts()

# Read PSB for channel, sum for SubID, and save

In [ ]:
PSB_DIR = pathlib.Path('../psb/')
PSB_CNT_DIR = PSB_DIR / f'count_no_cutoff'

In [ ]:
levels = 'class subclass subtype'.split()
with tqdm(total=df_obs[levels].nunique().sum()) as pbar:
    for level in levels:
        for ct in set(df_obs[level]):
            print(f'Processing {level}, {ct}                   ', end='\r')
            
            df_cnt = pd.read_csv(PSB_CNT_DIR / level / f'PAS_read_count_{ct}_{CURRENT_LEVEL}.csv.gz', index_col=0).T
            df_cnt[MERGED_LEVEL] = df_cnt.index.map(cur_2_merged)
            df_cnt_subid = df_cnt.groupby(MERGED_LEVEL).sum().T
            out_csv = PSB_CNT_DIR / level / f'PAS_read_count_{ct}_{MERGED_LEVEL}.csv'
            df_cnt_subid.to_csv(out_csv)
            subprocess.run(['pigz', '-f', out_csv])

            pbar.update(n=1)

In [ ]:
df_cnt[df_cnt[MERGED_LEVEL].isna()]